# DuckDB + Horizon Iceberg REST Catalog

Query Snowflake-managed silver Dynamic Iceberg Tables from DuckDB using Snowflake's
Horizon Iceberg REST Catalog (HIRC) and a Programmatic Access Token (PAT).

**Before running:** complete the Dynamic Iceberg Tables chapter so all five `dt_*` tables
exist in `balloon_silver.silver` and have refreshed at least once.

In [ ]:
# Configuration defaults — all values are read from .env first;
# these serve as fallbacks when the env vars are unset.
#
# Set in .env before running:
#   SNOWFLAKE_ACCOUNT_URL=https://<org>-<account>.snowflakecomputing.com
#   SA_USER=duckdb_sa
#   SA_ROLE=duckdb_silver_reader           # no hyphens (HIRC requirement)
#   SNOWFLAKE_SILVER_DATABASE=balloon_silver
# PAT is loaded automatically from the OS keyring (run: task snowflake:pat-create)
import os

__prefix = os.getenv("LAB_USERNAME")
if __prefix is None:
    __prefix = os.getenv("USER")
DEFAULT_SA_USER = f"{__prefix}_duckdb_sa"
DEFAULT_SA_ROLE = f"{__prefix}_duckdb_silver_reader"
DEFAULT_DATABASE = f"{__prefix}_balloon_silver"

print(f"DEFAULT_SA_USER:{DEFAULT_DATABASE!r}")
print(f"DEFAULT_SA_ROLE:{DEFAULT_DATABASE!r}")
print(f"DEFAULT_DATABASE:{DEFAULT_DATABASE!r}")

## Prerequisites

1. Silver DTs exist in `balloon_silver.SILVER` and have refreshed at least once.
2. Snowflake service account role `duckdb_silver_reader` has been created with SELECT
   on all silver DTs (see sfguide DuckDB Integration chapter).
3. Generate a PAT for `duckdb_sa` and store it in your OS keychain:
   ```bash
   task snowflake:pat-create
   ```
   The token is stored in the OS keyring automatically — no copy-paste needed.
4. `.env` (repo root) contains:
   ```
   SNOWFLAKE_ACCOUNT_URL=https://<org>-<account>.snowflakecomputing.com
   SA_USER=duckdb_sa
   SA_ROLE=duckdb_silver_reader
   SNOWFLAKE_SILVER_DATABASE=balloon_silver
   ```
5. Never commit `.env` — it is listed in `.gitignore`.

> **Role name rule:** HIRC does not support hyphens in role names.
> Use `duckdb_silver_reader` (underscores), not `duckdb-silver-reader`.


## Snowflake Setup (one-time)

Run the cell below once (as `ACCOUNTADMIN`) to grant the service account role the
minimum privileges needed to use HIRC.

| Grant | Why |
|---|---|
| `USAGE ON DATABASE` | HIRC catalog discovery (`/v1/config?warehouse=...`) |
| `USAGE ON SCHEMA` | Namespace listing (`/v1/namespaces`) |
| `SELECT ON ALL DYNAMIC TABLES` | Table reads via vended S3 credentials |
| `SELECT ON FUTURE DYNAMIC TABLES` | Auto-covers new DTs without re-granting |



> * **`GRANT SELECT ON ALL TABLES` silently skips Dynamic Tables** in Snowflake.
> Use `ON ALL DYNAMIC TABLES` and `ON FUTURE DYNAMIC TABLES` instead.
>
> * **HIRC case-sensitivity:** Polaris treats catalog names as case-sensitive.
> Snowflake stores database names in **uppercase**, so the `warehouse` parameter
> in the ATTACH call must also be uppercase (e.g. `BALLOON_SILVER`, not `balloon_silver`).
> A lowercase name returns `404 Not Found` on `/v1/config`.


In [ ]:
import subprocess
from dotenv import find_dotenv, load_dotenv
import os

load_dotenv(find_dotenv())

_db = os.getenv("SNOWFLAKE_SILVER_DATABASE", DEFAULT_DATABASE)
_role = os.getenv("SA_ROLE", DEFAULT_SA_ROLE)
_schema = "silver"

# Note: GRANT SELECT ON ALL TABLES silently skips Dynamic Tables in Snowflake.
# Use ON ALL DYNAMIC TABLES and ON FUTURE DYNAMIC TABLES instead.
_grants_sql = f"""
GRANT USAGE ON DATABASE {_db} TO ROLE {_role};
GRANT USAGE ON SCHEMA {_db}.{_schema} TO ROLE {_role};
GRANT SELECT ON ALL DYNAMIC TABLES IN SCHEMA {_db}.{_schema} TO ROLE {_role};
GRANT SELECT ON FUTURE DYNAMIC TABLES IN SCHEMA {_db}.{_schema} TO ROLE {_role};
"""

_result = subprocess.run(
    ["snow", "sql", "--query", _grants_sql, "--role", "ACCOUNTADMIN"],
    capture_output=True,
    text=True,
)
if _result.returncode != 0:
    print(f"Error applying grants:\n{_result.stderr}")
else:
    print(f"Grants applied for role {_role!r} on {_db}.{_schema}.")
    if _result.stdout.strip():
        print(_result.stdout)

In [ ]:
import duckdb
from dotenv import find_dotenv, load_dotenv
import os
import traceback

from sfutils_pat.pat import get_snowflake_connection_metadata
from sfutils_pat._keyring_store import load_pat

load_dotenv(find_dotenv())

sa_user = os.getenv("SA_USER", DEFAULT_SA_USER)
sa_role = os.getenv("SA_ROLE", DEFAULT_SA_ROLE)
database = os.getenv("SNOWFLAKE_SILVER_DATABASE", DEFAULT_DATABASE)
snowflake_account_url = os.getenv("SNOWFLAKE_ACCOUNT_URL", "").rstrip("/")

if not snowflake_account_url:
    raise ValueError("SNOWFLAKE_ACCOUNT_URL is not set — add it to .env")

# Load PAT from OS keyring; fall back to SNOWFLAKE_PASSWORD in .env
pat_token = None
pat_source = "env"
try:
    pat_name = os.getenv("PAT_NAME") or f"{sa_user}_pat".upper()
    account, host = get_snowflake_connection_metadata()
    pat_token = load_pat(host, account, sa_user, pat_name)
    if pat_token:
        pat_source = "keyring"
except Exception as e:
    print(f"Keyring lookup failed ({e}); falling back to SNOWFLAKE_PASSWORD env var")

pat_token = pat_token or os.getenv("SNOWFLAKE_PASSWORD")
if not pat_token:
    raise ValueError(f"No PAT found for {sa_user!r}. Run: task snowflake:pat-create")

catalog_uri = snowflake_account_url.lower() + "/polaris/api/catalog"

print(f"Catalog URI : {catalog_uri}")
print(f"Database    : {database}")
print(f"Role        : {sa_role}")
print(f"PAT source  : {pat_source}")

In [ ]:
# Install and load the DuckDB Iceberg and HTTPFS extensions.
# Extensions are cached after first install — subsequent runs are fast.
conn = duckdb.connect()
conn.execute("INSTALL iceberg;")
conn.execute("LOAD iceberg;")
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
print("Extensions loaded.")

## Connect to Horizon Iceberg REST Catalog

DuckDB authenticates via PAT using the OAuth2 client credentials flow, then attaches
the silver database. Snowflake vends temporary cloud credentials so DuckDB reads S3
data files directly — no data proxying through Snowflake.

> **Catalog name must be uppercase** (`database.upper()`). HIRC/Polaris is case-sensitive
> and the Snowflake database identifier is stored uppercase. The local DuckDB alias
> stays lowercase so subsequent queries use the shorter lowercase name.


In [ ]:
# HIRC (Polaris) is case-sensitive: warehouse name must be uppercase.
# The local DuckDB alias is kept lowercase for query convenience.
catalog_name = database.upper()

secret_sql = f"""
  CREATE OR REPLACE SECRET iceberg_pat_secret (
    TYPE iceberg,
    CLIENT_ID '',
    CLIENT_SECRET '{pat_token}',
    OAUTH2_SERVER_URI '{catalog_uri}/v1/oauth/tokens',
    OAUTH2_GRANT_TYPE 'client_credentials',
    OAUTH2_SCOPE 'session:role:{sa_role}'
  );
"""

attach_sql = f"""
  ATTACH '{catalog_name}' AS {database} (
    TYPE iceberg,
    SECRET iceberg_pat_secret,
    ENDPOINT '{catalog_uri}',
    SUPPORT_NESTED_NAMESPACES false
  );
"""

try:
    conn.execute(secret_sql)
    conn.execute(attach_sql)
    print(f"Attached {catalog_name} (alias: {database}) via HIRC.")
except Exception:
    traceback.print_exc()

## Discover Tables

Snowflake identifiers are **UPPERCASE** when accessed through HIRC.
Use `SILVER.DT_PLAYER_LEADERBOARD` (uppercase), not `silver.dt_player_leaderboard`.

In [ ]:
try:
    conn.execute(f"USE {database}.SILVER")
    _rows = conn.execute("SHOW TABLES").fetchall()
    print(f"Found {len(_rows)} table(s) in {database}.SILVER:")
    for _r in _rows:
        print(f"  {database}.SILVER.{_r[0]}")
except Exception:
    traceback.print_exc()

## Query Silver Dynamic Iceberg Tables

Each cell queries one of the five silver DTs. All identifiers use uppercase schema
and table names as required by HIRC.

In [ ]:
# dt_player_leaderboard — per-player total score and bonus pops
try:
    df = conn.execute(f"""
        SELECT player, total_score, bonus_pops, last_event_ts
        FROM {database.upper()}.SILVER.DT_PLAYER_LEADERBOARD
        ORDER BY total_score DESC NULLS LAST
        LIMIT 10
    """).df()
    print("dt_player_leaderboard:")
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_balloon_color_stats — per-player, per-color breakdown
try:
    df = conn.execute(f"""
        SELECT player, balloon_color, balloon_pops, points_by_color, bonus_hits
        FROM {database}.SILVER.DT_BALLOON_COLOR_STATS
        ORDER BY player, points_by_color DESC NULLS LAST
        LIMIT 10
    """).df()
    print("dt_balloon_color_stats:")
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_realtime_scores — 15-second windowed scores per player
try:
    df = conn.execute(f"""
        SELECT player, total_score, window_start, window_end
        FROM {database}.SILVER.DT_REALTIME_SCORES
        ORDER BY window_start DESC, player
        LIMIT 10
    """).df()
    print("dt_realtime_scores:")
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_balloon_colored_pops — 15-second windows by player and balloon color
try:
    df = conn.execute(f"""
        SELECT player, balloon_color, balloon_pops, window_start, window_end
        FROM {database}.SILVER.DT_BALLOON_COLORED_POPS
        ORDER BY window_start DESC, player, balloon_color
        LIMIT 10
    """).df()
    print("dt_balloon_colored_pops:")
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()

In [ ]:
# dt_color_performance_trends — avg score per pop by color over 15-second windows
try:
    df = conn.execute(f"""
        SELECT balloon_color, avg_score_per_pop, total_pops, window_start, window_end
        FROM {database}.SILVER.DT_COLOR_PERFORMANCE_TRENDS
        ORDER BY window_start DESC, avg_score_per_pop DESC NULLS LAST
        LIMIT 10
    """).df()
    print("dt_color_performance_trends:")
    print(df.to_string(index=False))
except Exception:
    traceback.print_exc()